#### Environment Setup & API Keys

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [31]:


from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import TokenTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.chains.retrieval_qa.base import RetrievalQA
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

In [4]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.3, api_key=os.getenv("GROQ_API_KEY"))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2708.51it/s]


#### PDF Document Loading

In [6]:
file_path="data/SDG.pdf"
loder= PyPDFLoader(file_path)
data=loder.load()


In [7]:
data

[Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign CC 13.0 (Macintosh)', 'creationdate': '2017-12-26T16:03:25-05:00', 'moddate': '2017-12-27T15:29:10-05:00', 'trapped': '/False', 'source': 'data/SDG.pdf', 'total_pages': 24, 'page': 0, 'page_label': '1'}, page_content=''),
 Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign CC 13.0 (Macintosh)', 'creationdate': '2017-12-26T16:03:25-05:00', 'moddate': '2017-12-27T15:29:10-05:00', 'trapped': '/False', 'source': 'data/SDG.pdf', 'total_pages': 24, 'page': 1, 'page_label': '2'}, page_content=''),
 Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign CC 13.0 (Macintosh)', 'creationdate': '2017-12-26T16:03:25-05:00', 'moddate': '2017-12-27T15:29:10-05:00', 'trapped': '/False', 'source': 'data/SDG.pdf', 'total_pages': 24, 'page': 2, 'page_label': '3'}, page_content='IN THE YEAR 2015, LEADERS FROM 193 COUNTRIES OF THE WORLD \nCAME TOGETHER TO FACE T

#### Text Extraction for Question Generation

In [8]:
question_gen=""
for page in data:
  question_gen+=page.page_content

In [9]:
question_gen

'IN THE YEAR 2015, LEADERS FROM 193 COUNTRIES OF THE WORLD \nCAME TOGETHER TO FACE THE FUTURE.\nAnd what they saw was daunting. Famines. Drought. Wars. Plagues. Poverty. \nNot just in some faraway place, but in their own cities and towns and villages.\nThey knew things didn’t have to be this way. They knew we had enough \nfood to feed the world, but that it wasn’t getting shared. They knew there \nwere medicines for HIV and other diseases, but they cost a lot. They knew \nthat earthquakes and floods were inevitable, but that the high death \ntolls were not. \nThey also knew that billions of people worldwide shared their hope for a \nbetter future.\nSo leaders from these countries created a plan called the Sustainable \nDevelopment Goals (SDGs). This set of 17 goals imagines a future just 15 years \noff that would be rid of poverty and hunger, and safe from the worst effects of \nclimate change. It’s an ambitious plan. \nBut there’s ample evidence that we can succeed. In the past 15 yea

#### Token Splitting (Large Chunks for Questions)

In [10]:
splitter_ques_gen=TokenTextSplitter(
  model_name="gpt-3.5-turbo",
  chunk_size=10000,
  chunk_overlap=200
)

In [11]:
chunk_question_gen=splitter_ques_gen.split_text(question_gen)

In [12]:
chunk_question_gen

['IN THE YEAR 2015, LEADERS FROM 193 COUNTRIES OF THE WORLD \nCAME TOGETHER TO FACE THE FUTURE.\nAnd what they saw was daunting. Famines. Drought. Wars. Plagues. Poverty. \nNot just in some faraway place, but in their own cities and towns and villages.\nThey knew things didn’t have to be this way. They knew we had enough \nfood to feed the world, but that it wasn’t getting shared. They knew there \nwere medicines for HIV and other diseases, but they cost a lot. They knew \nthat earthquakes and floods were inevitable, but that the high death \ntolls were not. \nThey also knew that billions of people worldwide shared their hope for a \nbetter future.\nSo leaders from these countries created a plan called the Sustainable \nDevelopment Goals (SDGs). This set of 17 goals imagines a future just 15 years \noff that would be rid of poverty and hunger, and safe from the worst effects of \nclimate change. It’s an ambitious plan. \nBut there’s ample evidence that we can succeed. In the past 15 ye

In [13]:
type(chunk_question_gen[0])

str

In [14]:
from langchain_core.documents import Document


#### String Chunks to Document Objects Conversion

In [15]:
document_ques_gen=[Document(page_content=t) for t in chunk_question_gen]
document_ques_gen

[Document(metadata={}, page_content='IN THE YEAR 2015, LEADERS FROM 193 COUNTRIES OF THE WORLD \nCAME TOGETHER TO FACE THE FUTURE.\nAnd what they saw was daunting. Famines. Drought. Wars. Plagues. Poverty. \nNot just in some faraway place, but in their own cities and towns and villages.\nThey knew things didn’t have to be this way. They knew we had enough \nfood to feed the world, but that it wasn’t getting shared. They knew there \nwere medicines for HIV and other diseases, but they cost a lot. They knew \nthat earthquakes and floods were inevitable, but that the high death \ntolls were not. \nThey also knew that billions of people worldwide shared their hope for a \nbetter future.\nSo leaders from these countries created a plan called the Sustainable \nDevelopment Goals (SDGs). This set of 17 goals imagines a future just 15 years \noff that would be rid of poverty and hunger, and safe from the worst effects of \nclimate change. It’s an ambitious plan. \nBut there’s ample evidence tha

In [16]:
type(document_ques_gen[0])

langchain_core.documents.base.Document

In [17]:
splitter_ans_gen=TokenTextSplitter(
  model_name="gpt-3.5-turbo",
  chunk_size=1000,
  chunk_overlap=100
)

#### Smaller Chunks for Answer Generation VectorStore

In [18]:
document_ans_gen=splitter_ans_gen.split_documents(
  document_ques_gen
)

In [19]:
document_ans_gen

[Document(metadata={}, page_content='IN THE YEAR 2015, LEADERS FROM 193 COUNTRIES OF THE WORLD \nCAME TOGETHER TO FACE THE FUTURE.\nAnd what they saw was daunting. Famines. Drought. Wars. Plagues. Poverty. \nNot just in some faraway place, but in their own cities and towns and villages.\nThey knew things didn’t have to be this way. They knew we had enough \nfood to feed the world, but that it wasn’t getting shared. They knew there \nwere medicines for HIV and other diseases, but they cost a lot. They knew \nthat earthquakes and floods were inevitable, but that the high death \ntolls were not. \nThey also knew that billions of people worldwide shared their hope for a \nbetter future.\nSo leaders from these countries created a plan called the Sustainable \nDevelopment Goals (SDGs). This set of 17 goals imagines a future just 15 years \noff that would be rid of poverty and hunger, and safe from the worst effects of \nclimate change. It’s an ambitious plan. \nBut there’s ample evidence tha

In [20]:
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.3, api_key=os.getenv("GROQ_API_KEY"))
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2022.99it/s]


In [21]:
prompt_template = """
You are an expert at creating questions based on coding materials and documentation.
Your goal is to prepare a coder or programmer for their exam and coding tests.
You do this by asking questions about the text below:

------------
{text}
------------

Create questions that will prepare the coders or programmers
Make sure not to lose any important information.

QUESTIONS:
"""

In [22]:
prompt_questions=PromptTemplate(template=prompt_template,input_variables=['text'])

In [23]:
refine_template = ("""
You are an expert at creating practice questions based on coding material and documentation.
Your goal is to help a coder or programmer prepare for a coding test.
We have received some practice questions to a certain extent: {existing_answer}.
We have the option to refine the existing questions or add new ones
(only if necessary) with some more context below.
----------
{text}
----------

Given the new context, refine the original questions in English.
If the context is not helpful, please provide the original questions.
QUESTIONS:
"""
)


In [24]:
refine_prompt_questions=PromptTemplate(template=refine_template,input_variables=["existing_answer","text"])

In [26]:
from langchain_classic.chains.summarize import load_summarize_chain

In [27]:
ques_gen_chain=load_summarize_chain(llm=llm,chain_type="refine",verbose=True,question_prompt=prompt_questions,refine_prompt=refine_prompt_questions)

In [28]:
ques=ques_gen_chain.run(document_ques_gen)
print(ques)

C:\Users\impav\AppData\Local\Temp\ipykernel_5416\137075838.py:1: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  ques=ques_gen_chain.run(document_ques_gen)




> Entering new RefineDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:

You are an expert at creating questions based on coding materials and documentation.
Your goal is to prepare a coder or programmer for their exam and coding tests.
You do this by asking questions about the text below:

------------
IN THE YEAR 2015, LEADERS FROM 193 COUNTRIES OF THE WORLD 
CAME TOGETHER TO FACE THE FUTURE.
And what they saw was daunting. Famines. Drought. Wars. Plagues. Poverty. 
Not just in some faraway place, but in their own cities and towns and villages.
They knew things didn’t have to be this way. They knew we had enough 
food to feed the world, but that it wasn’t getting shared. They knew there 
were medicines for HIV and other diseases, but they cost a lot. They knew 
that earthquakes and floods were inevitable, but that the high death 
tolls were not. 
They also knew that billions of people worldwide shared their hope for a 
better future.
So leaders from

In [34]:
vectorstore=FAISS.from_documents(document_ans_gen,embeddings)

In [42]:
ques

"Here are some questions based on the provided text to help prepare coders or programmers for their exam and coding tests:\n\n**Section 1: Introduction to Sustainable Development Goals (SDGs)**\n\n1. What year did leaders from 193 countries come together to face the future and create the Sustainable Development Goals (SDGs)?\n2. What are the 17 goals of the SDGs, and what is the expected outcome by 2030?\n3. What is the role of the United Nations Development Programme (UNDP) in achieving the SDGs?\n\n**Section 2: SDG 1 - End Poverty**\n\n1. What is the current number of people living in extreme poverty worldwide, and what is the goal to achieve by 2030?\n2. How has the international community made progress in reducing extreme poverty in the past 15 years?\n3. What are some strategies to end poverty altogether, and how can individuals contribute to this effort?\n\n**Section 3: SDG 2 - End Hunger**\n\n1. What progress has been made in reducing hunger worldwide in the past 20 years?\n2. W

In [44]:
import re
questions_list = re.findall(r"^\d+\.\s*(.*)", ques, flags=re.MULTILINE)

In [45]:
questions_list

['What year did leaders from 193 countries come together to face the future and create the Sustainable Development Goals (SDGs)?',
 'What are the 17 goals of the SDGs, and what is the expected outcome by 2030?',
 'What is the role of the United Nations Development Programme (UNDP) in achieving the SDGs?',
 'What is the current number of people living in extreme poverty worldwide, and what is the goal to achieve by 2030?',
 'How has the international community made progress in reducing extreme poverty in the past 15 years?',
 'What are some strategies to end poverty altogether, and how can individuals contribute to this effort?',
 'What progress has been made in reducing hunger worldwide in the past 20 years?',
 'What are some ways to promote sustainable agriculture and support small farmers to achieve zero hunger?',
 'How can individuals contribute to ending hunger and malnutrition by 2030?',
 'What are some significant health achievements made in the past 25 years, and what are the re

In [35]:
answer_generation_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever()
)

In [ ]:
for question in questions_list:
  print("Question: ",question)
  ans=answer_generation_chain.run(question)
  print("Answer: ",ans)
  print("********")

  with open("answers.txt","a") as f:
    f.write("Question: "+ question + "\\n")
    f.write("**********")

Question:  What year did leaders from 193 countries come together to face the future and create the Sustainable Development Goals (SDGs)?
Answer:  The year was 2015. Leaders from 193 countries came together to create the Sustainable Development Goals (SDGs).
********
Question:  What are the 17 goals of the SDGs, and what is the expected outcome by 2030?
Answer:  The 17 Sustainable Development Goals (SDGs) are:

1. **End Poverty**: End poverty in all its forms everywhere.
2. **Zero Hunger**: End hunger, achieve food security and improved nutrition, and promote sustainable agriculture.
3. **Good Health and Well-being**: Ensure healthy lives and promote well-being for all at all ages.
4. **Quality Education**: Ensure inclusive and equitable quality education and promote lifelong learning opportunities for all.
5. **Gender Equality**: Achieve gender equality and empower all women and girls.
6. **Clean Water and Sanitation**: Ensure availability and sustainable management of water and sanit